# 11. Tests and final results

Runs the project's tests, then shows the saved benchmark results and what is now stored in the lake.

## What to look at

- The **test table**: every test should say `passed` (tests that need the optional translation step may say `skipped`).
- **Results**: copied from the saved files in `benchmarks/`, not typed.
- The **storage table**: size of each table in MinIO.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
stop_spark()   # the tests start their own Spark session in a separate process
junit = ROOT / "benchmarks" / ".junit.xml"
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "tests", f"--junitxml={junit}"], capture_output=True, text=True, cwd=ROOT)
print(r.stdout.strip().splitlines()[-1])
import xml.etree.ElementTree as ET
rows = []
for tc in ET.parse(junit).getroot().iter("testcase"):
    outcome = "failed" if tc.find("failure") is not None or tc.find("error") is not None else ("skipped" if tc.find("skipped") is not None else "passed")
    rows.append((tc.get("name"), outcome, round(float(tc.get("time")), 1)))
junit.unlink()
tests = pd.DataFrame(rows, columns=["test", "outcome", "seconds"])
display(tests.groupby("outcome").size().rename("tests").reset_index())
assert r.returncode == 0, r.stdout[-2000:]
tests

In [ ]:
e = json.loads((BENCH_DIR / "embedding_benchmark.json").read_text())
print(f"Embeddings: batched pandas_udf was {e['speedup_batched_vs_naive']}x the naive UDF ({e['batched_rows_per_sec']:.0f} vs {e['naive_rows_per_sec']:.0f} rows/s over {e['rows']} rows).")
display(Markdown((BENCH_DIR / "ivf_benchmark.md").read_text()))

In [ ]:
q = BENCH_DIR / "qdrant_comparison.md"
display(Markdown(q.read_text())) if q.exists() else print("Qdrant bonus not run (make bonus).")

In [ ]:
minio_summary()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Open `warehouse/`. The whole lakehouse is there: `bronze/reviews`, `silver/review_embeddings` (and `silver/review_translations` if you ran 05b), `gold/centroids` and `gold/review_embeddings`, each with a `data/` and a `metadata/` folder. The storage table above is the same listing as numbers.